# GLTC-TNN 图像修复算法
几何低秩张量补全 - 截断核范数 (Geometric Low-rank Tensor Completion - Truncated Nuclear Norm)

In [ ]:
import numpy as np
from numpy.linalg import inv
from skimage.metrics import structural_similarity as ssim
from PIL import Image
import matplotlib.pyplot as plt
import os

def ten2mat(tensor, mode):
    """张量按模展开为矩阵"""
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order='F')

def mat2ten(mat, tensor_size, mode):
    """矩阵折叠为张量"""
    index = list()
    index.append(mode)
    for i in range(tensor_size.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(tensor_size[index]), order='F'), 0, mode)

def svt_tnn(mat, tau, theta):
    """截断核范数的奇异值阈值化"""
    u, s, v = np.linalg.svd(mat, full_matrices=0)
    vec = np.zeros(len(s))
    for i in range(len(s)):
        if i >= theta:
            vec[i] = s[i] - tau
        else:
            vec[i] = s[i]
    vec[vec < 0] = 0
    return np.matmul(np.matmul(u, np.diag(vec)), v)

In [ ]:
def GLTC_TNN(dense_tensor, sparse_tensor, alpha, beta, rho, theta, maxiter, verbose=True):
    """
    GLTC-TNN 算法主函数
    
    参数:
    dense_tensor: 原始完整图像 (用于计算误差)
    sparse_tensor: 观测到的稀疏图像
    alpha: 正则化参数
    beta: 图正则化参数
    rho: ADMM惩罚参数
    theta: 截断参数 (保留前theta个奇异值)
    maxiter: 最大迭代次数
    verbose: 是否打印进度
    
    返回:
    tensor_hat: 修复后的张量
    psnr_list: 每次迭代的PSNR值
    ssim_value: 最终SSIM值
    """
    dim0 = sparse_tensor.ndim
    dim1, dim2, dim3 = sparse_tensor.shape
    dim = np.array([dim1, dim2, dim3])
    maxP = float(np.max(dense_tensor))
    
    # 创建二值掩码
    binary_tensor = np.zeros((dim1, dim2, dim3))
    binary_tensor[np.where(sparse_tensor != 0)] = 1
    tensor_hat = sparse_tensor.copy()
    
    # 初始化变量
    X = np.zeros((dim1, dim2, dim3, dim0))
    Z = np.zeros((dim1, dim2, dim3, dim0))
    T = np.zeros((dim1, dim2, dim3, dim0))
    for k in range(dim0):
        X[:, :, :, k] = tensor_hat
        Z[:, :, :, k] = tensor_hat
    
    # 构造邻接平滑矩阵
    D1 = np.zeros((dim1 - 1, dim1))
    for i in range(dim1 - 1):
        D1[i, i] = -1
        D1[i, i + 1] = 1
    D2 = np.zeros((dim2 - 1, dim2))
    for i in range(dim2 - 1):
        D2[i, i] = -1
        D2[i, i + 1] = 1
    
    # 缺失位置
    pos = np.where((dense_tensor != 0) & (sparse_tensor == 0))
    psnr_list = np.zeros(maxiter)
    
    for iters in range(maxiter):
        # 更新Z
        for k in range(dim0):
            Z_hat = ten2mat(X[:, :, :, k] + T[:, :, :, k] / rho, k)
            Z_hat = svt_tnn(Z_hat, alpha / rho, theta)
            Z[:, :, :, k] = mat2ten(Z_hat, dim, k)
            
            # 更新X
            var = ten2mat(rho * Z[:, :, :, k] - T[:, :, :, k], k)
            if k == 0:
                var0 = mat2ten(np.matmul(inv(beta * np.matmul(D1.T, D1) + rho * np.eye(dim1)), var), dim, k)
            elif k == 1:
                var0 = mat2ten(np.matmul(inv(beta * np.matmul(D2.T, D2) + rho * np.eye(dim2)), var), dim, k)
            else:
                var0 = Z[:, :, :, k] - T[:, :, :, k] / rho
            X[:, :, :, k] = np.multiply(1 - binary_tensor, var0) + sparse_tensor
        
        # 平均
        tensor_hat = np.mean(X, axis=3)
        
        # 限制范围
        tensor_hat[np.where(tensor_hat > maxP)] = maxP
        tensor_hat[np.where(tensor_hat < 0)] = 0
        
        # 计算PSNR
        mseC1 = np.linalg.norm(dense_tensor[:, :, 0].astype(float) - np.round(tensor_hat[:, :, 0]), 'fro') ** 2 / (dim1 * dim2)
        psnrC1 = 10 * np.log10(maxP**2 / mseC1)
        mseC2 = np.linalg.norm(dense_tensor[:, :, 1].astype(float) - np.round(tensor_hat[:, :, 1]), 'fro') ** 2 / (dim1 * dim2)
        psnrC2 = 10 * np.log10(maxP**2 / mseC2)
        mseC3 = np.linalg.norm(dense_tensor[:, :, 2].astype(float) - np.round(tensor_hat[:, :, 2]), 'fro') ** 2 / (dim1 * dim2)
        psnrC3 = 10 * np.log10(maxP**2 / mseC3)
        psnr_list[iters] = (psnrC1 + psnrC2 + psnrC3) / 3
        
        if verbose:
            print(f"Epoch = {iters+1}, PSNR = {psnr_list[iters]:.4f} dB")
        
        # 更新拉格朗日乘子和X
        for k in range(dim0):
            T[:, :, :, k] = T[:, :, :, k] + rho * (X[:, :, :, k] - Z[:, :, :, k])
            X[:, :, :, k] = tensor_hat.copy()
    
    # 计算最终SSIM
    ssim_value = ssim(dense_tensor.astype(float), tensor_hat, 
                      data_range=maxP, 
                      channel_axis=2)
    
    if verbose:
        print(f"\n{'='*50}")
        print(f"最终PSNR: {psnr_list[-1]:.4f} dB")
        print(f"最终SSIM: {ssim_value:.4f}")
        print(f"{'='*50}")
    
    return tensor_hat, psnr_list, ssim_value

In [ ]:
# 参数设置
import time
import pandas as pd

seedr = 920
np.random.seed(seedr)

# 图像列表
image_names = ['Airplane', 'House256', 'House512', 'Peppers', 'Tree', 'Sailboat', 'Female']

# 算法参数
missing_rates = [0.8, 0.9, 0.95]
theta = 100
alpha = 10
rho = 1.001
beta = 0.1 * rho
maxiter = 1000

# 按缺失率批量处理
all_results = {}

for missing_rate in missing_rates:
    result_dir = f'results/GLTC-TNN/GLTC-TNN{missing_rate*100}%'
    os.makedirs(result_dir, exist_ok=True)
    results = {}

    print(f"\n{'#'*70}")
    print(f"开始处理缺失率: {missing_rate*100:.1f}%")
    print(f"{'#'*70}\n")

    for img_name in image_names:
        print(f"\n{'='*60}")
        print(f"处理图像: {img_name}")
        print(f"{'='*60}\n")

        try:
            image = np.array(Image.open(f'./data/{img_name}.tiff'))
            h, w, c = image.shape

            mask = np.random.rand(h, w) > missing_rate
            sparse_image = image.copy()
            for k in range(c):
                sparse_image[:, :, k] = np.multiply(image[:, :, k], mask)

            print(f"图像尺寸: {image.shape}")
            print(f"缺失率: {missing_rate*100:.1f}%")
            print(f"观测像素: {np.sum(sparse_image > 0)} / {image.size}\n")

            start_time = time.perf_counter()
            image_hat, psnr_list, ssim_value = GLTC_TNN(image, sparse_image, alpha, beta, rho, theta, maxiter)
            elapsed_time = time.perf_counter() - start_time

            image_rec = np.round(image_hat).astype(int)
            image_rec[np.where(image_rec > 255)] = 255
            image_rec[np.where(image_rec < 0)] = 0

            print(f"运行时间: {elapsed_time:.2f} 秒")

            results[img_name] = {
                'original': image,
                'observed': sparse_image,
                'recovered': image_rec,
                'psnr_list': psnr_list,
                'final_psnr': psnr_list[-1],
                'ssim': ssim_value,
                'time': elapsed_time
            }

            Image.fromarray(np.uint8(image_rec)).save(f'{result_dir}/{img_name}_修复后.png')

        except Exception as e:
            print(f"处理 {img_name} 时出错: {str(e)}")
            continue

    plt.rcParams['font.sans-serif'] = ['SimHei']

    for img_name in results.keys():
        data = results[img_name]

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(np.uint8(data['original']))
        axes[0].set_title('原始图像', fontsize=14)
        axes[0].axis('off')

        axes[1].imshow(np.uint8(data['observed']))
        axes[1].set_title(f'观测数据 ({missing_rate*100:.0f}% 缺失)', fontsize=14)
        axes[1].axis('off')

        axes[2].imshow(np.uint8(data['recovered']))
        axes[2].set_title(
            f'GLTC-TNN 修复\nPSNR={data["final_psnr"]:.2f}dB, SSIM={data["ssim"]:.4f}\n时间={data["time"]:.2f}s',
            fontsize=14
        )
        axes[2].axis('off')

        plt.tight_layout()
        plt.savefig(f'{result_dir}/{img_name}_对比.png', dpi=150, bbox_inches='tight')
        plt.show()

        plt.figure(figsize=(10, 6))
        plt.plot(range(1, len(data['psnr_list']) + 1), data['psnr_list'], 'b-', linewidth=2)
        plt.xlabel('迭代次数', fontsize=12)
        plt.ylabel('PSNR (dB)', fontsize=12)
        plt.title(f'{img_name} - GLTC-TNN 收敛曲线', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{result_dir}/{img_name}_收敛曲线.png', dpi=150, bbox_inches='tight')
        plt.show()

    summary_data = []
    for img_name, data in results.items():
        summary_data.append({
            '图像名称': img_name,
            'PSNR (dB)': f"{data['final_psnr']:.4f}",
            'SSIM': f"{data['ssim']:.4f}",
            '运行时间 (s)': f"{data['time']:.2f}"
        })

    df = pd.DataFrame(summary_data)
    df.to_excel(f'{result_dir}/实验总结.xlsx', index=False, engine='openpyxl')

    print(f"\n缺失率 {missing_rate*100:.1f}% 的结果已保存")
    print(f"Excel 已导出到: {result_dir}/实验总结.xlsx")

    all_results[missing_rate] = results

print(f"\n{'='*60}")
print('所有缺失率处理完成！')
print(f"{'='*60}")


In [ ]:
# 结果已经在上一单元按缺失率分别保存
import pandas as pd

if 'all_results' not in globals():
    raise NameError('当前内核中没有 all_results，请先确保实验结果还保留在内存里。')

for missing_rate in missing_rates:
    result_dir = f'results/GLTC-TNN/GLTC-TNN{missing_rate*100}%'
    results = all_results[missing_rate]

    summary_data = []
    for img_name, data in results.items():
        summary_data.append({
            '图像名称': img_name,
            'PSNR (dB)': f"{data['final_psnr']:.4f}",
            'SSIM': f"{data['ssim']:.4f}",
            '运行时间 (s)': f"{data['time']:.2f}"
        })

    df = pd.DataFrame(summary_data)
    df.to_excel(f'{result_dir}/实验总结.xlsx', index=False, engine='openpyxl')
    print(f'缺失率 {missing_rate*100:.1f}% 的 Excel 已导出到: {result_dir}/实验总结.xlsx')